# Milestone 2 & 3: Baseline CNN and INT8 Quantization
This notebook trains a 1D Convolutional Neural Network on the extracted MIT-BIH 
heartbeat windows, and then compresses it using TFLite Post-Training Quantization.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
import os
import time

### 1. Load the Preprocessed Dataset
We load the arrays generated by the preprocessing script.

In [ ]:
print("Loading preprocessed dataset...")
X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

### 2. Build the 1D CNN Architecture
We design a lightweight architecture specifically suited for Edge deployment.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(180, 1)),
    tf.keras.layers.Conv1D(16, kernel_size=7, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

### 3. Handle Class Imbalance & Train
Normal beats heavily outnumber Abnormal beats. We compute class weights to balance the loss.

In [ ]:
neg, pos = np.bincount(y_train)
total = neg + pos
class_weight = {0: (1 / neg)*(total/2.0), 1: (1 / pos)*(total/2.0)}

print("Starting Training (5 Epochs)...")
history = model.fit(
    X_train, y_train, 
    epochs=5, 
    batch_size=128, 
    validation_split=0.1, 
    class_weight=class_weight, 
    verbose=1
)

# Save Baseline Model
os.makedirs('../models', exist_ok=True)
model.save('../models/baseline_cnn.keras')
print(f"\nBaseline Model Size: {os.path.getsize('../models/baseline_cnn.keras') / 1024:.2f} KB")

### 4. INT8 Post-Training Quantization
We convert the model to TensorFlow Lite and quantize all weights and activations 
to 8-bit integers to fit on a microcontroller.

In [ ]:
def representative_data_gen():
    # Use 500 samples from the training set to calibrate the quantization ranges
    for input_value in tf.data.Dataset.from_tensor_slices(X_train).batch(1).take(500):
        yield [tf.cast(input_value, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Enforce fully INT8 quantization
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

with open('../models/model_quantized.tflite', 'wb') as f:
    f.write(tflite_model_quant)

### 5. Compression Results
Let's compare the size of the original floating-point model versus the quantized edge model.

In [ ]:
fp32_size = os.path.getsize('../models/baseline_cnn.keras')
int8_size = os.path.getsize('../models/model_quantized.tflite')

print("\n=== Size Comparison ===")
print(f"FP32 Model Size : {fp32_size / 1024:.2f} KB")
print(f"INT8 Model Size : {int8_size / 1024:.2f} KB")
print(f"Compression Ratio: {fp32_size / int8_size:.2f}x smaller!")